In [ ]:

import pandas as pd

# 读取训练集数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv'
train_df = pd.read_csv(train_data_path)

# 显示数据的前几行
print(train_df.head())


      id  Gender  ...                 MTRANS           NObeyesdad
0   9958    Male  ...             Automobile       Obesity_Type_I
1   7841    Male  ...  Public_Transportation  Insufficient_Weight
2   9293    Male  ...  Public_Transportation      Obesity_Type_II
3  15209  Female  ...             Automobile       Obesity_Type_I
4  16515    Male  ...  Public_Transportation  Overweight_Level_II

[5 rows x 18 columns]


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 读取训练集数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv'
train_df = pd.read_csv(train_data_path)

# 检查数据是否有缺失值
print(train_df.isnull().sum())

# 编码分类变量
label_encoders = {}
for column in train_df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    train_df[column] = le.fit_transform(train_df[column])
    label_encoders[column] = le

# 划分数据集
X = train_df.drop(columns=['id', 'NObeyesdad'])
y = train_df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


id                                0
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64
Training set size: 13284
Testing set size: 3322


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 初始化随机森林分类器
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# 训练模型
clf.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = clf.predict(X_test)

# 计算准确性
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set: {accuracy:.4f}")



Model accuracy on test set: 0.8934


In [ ]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectFromModel

# 初始化随机森林分类器
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# 训练模型
clf.fit(X_train, y_train)

# 特征选择
selector = SelectFromModel(clf, prefit=True)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# 训练新的随机森林分类器
clf.fit(X_train_selected, y_train)

# 在测试集上进行预测
y_pred = clf.predict(X_test_selected)

# 计算准确性
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set with selected features: {accuracy:.4f}")



D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
Model accuracy on test set with selected features: 0.8736


In [ ]:


from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectFromModel

# 初始化随机森林分类器
clf = RandomForestClassifier(random_state=42)

# 超参数网格
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 网格搜索
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# 最佳参数
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")

# 使用最佳参数训练模型
best_clf = RandomForestClassifier(**best_params, random_state=42)
best_clf.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = best_clf.predict(X_test)

# 计算准确性
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set with best parameters: {accuracy:.4f}")



Best parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Model accuracy on test set with best parameters: 0.8995


In [ ]:



from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 初始化随机森林分类器
clf = RandomForestClassifier(random_state=42)

# 超参数网格
param_grid = {
    'n_estimators': [100],
    'max_depth': [None],
    'min_samples_split': [2],
    'min_samples_leaf': [2],
    'bootstrap': [True, False]
}

# 特征缩放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 网格搜索
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# 最佳参数
best_params = grid_search.best_params_
print(f"Best parameters with feature scaling: {best_params}")

# 使用最佳参数训练模型
best_clf = RandomForestClassifier(**best_params, random_state=42)
best_clf.fit(X_train_scaled, y_train)

# 在测试集上进行预测
y_pred = best_clf.predict(X_test_scaled)

# 计算准确性
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set with feature scaling and best parameters: {accuracy:.4f}")



Best parameters with feature scaling: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Model accuracy on test set with feature scaling and best parameters: 0.8998
